In [ ]:
# Core Imports and Environment Setup
# Automatically installs missing dependencies if needed

import sys
import os
import subprocess
from pathlib import Path
import time


def setup_environment_and_check_dependencies():
    """
    Set up environment, check dependencies, and verify external tools.
    Handles virtual environment creation, package installation, and tool detection.
    
    Returns:
    - bool: True if setup completed successfully, False otherwise
    """
    # --- Environment Report ---
    divider = "=" * 60
    print(f"{divider}\nEnvironment Check\n{divider}")
    print(f"Python        : {sys.version.split()[0]}")
    print(f"Working dir   : {os.getcwd()}")

    # Check for virtual environment
    venv_exists = Path(".venv").exists()
    if not venv_exists:
        print("⚠ Virtual environment (.venv) not found")
        print("   Creating virtual environment and installing dependencies...")
        try:
            # Run install_dependencies.sh
            result = subprocess.run(
                ["bash", "install_dependencies.sh"],
                capture_output=True,
                text=True,
                timeout=600  # 10 minute timeout
            )
            if result.returncode == 0:
                print("✓ Virtual environment created and dependencies installed")
                print("   Please restart the kernel to use the new environment")
            else:
                print(f"⚠ Installation had issues:\n{result.stderr[:500]}")
                print("   Please run manually: ./install_dependencies.sh")
        except FileNotFoundError:
            print("⚠ install_dependencies.sh not found")
            print("   Creating virtual environment manually...")
            subprocess.run([sys.executable, "-m", "venv", ".venv"], check=False)
        except subprocess.TimeoutExpired:
            print("⚠ Installation timed out")
            print("   Please run manually: ./install_dependencies.sh")
        except Exception as e:
            print(f"⚠ Error during installation: {e}")
            print("   Please run manually: ./install_dependencies.sh")

    # Core package check and auto-install
    missing = []
    for pkg in ["Bio", "numpy", "requests"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print(f"\n⚠ Missing packages: {', '.join(missing)}")
        print("   Attempting to install missing packages...")
        
        # Determine pip command (use venv pip if available)
        pip_cmd = [sys.executable, "-m", "pip"]
        venv_pip = Path(".venv/bin/pip")
        if venv_pip.exists():
            pip_cmd = [str(venv_pip)]
            print("   Using virtual environment pip...")
        
        # Try to install via pip
        try:
            for pkg in missing:
                if pkg == "Bio":
                    pkg_name = "biopython"
                else:
                    pkg_name = pkg
                
                print(f"   Installing {pkg_name}...")
                result = subprocess.run(
                    pip_cmd + ["install", pkg_name, "--quiet"],
                    capture_output=True,
                    text=True,
                    timeout=300
                )
                if result.returncode == 0:
                    print(f"   ✓ {pkg_name} installed")
                else:
                    print(f"   ⚠ Failed to install {pkg_name}")
                    if result.stderr:
                        print(f"   Error: {result.stderr[:200]}")
        except Exception as e:
            print(f"   ⚠ Error installing packages: {e}")
            print("   Please run: ./install_dependencies.sh")
        
        # Re-check after installation attempt
        still_missing = []
        for pkg in missing:
            try:
                __import__(pkg)
            except ImportError:
                still_missing.append(pkg)
        
        if still_missing:
            print(f"\n⚠ Still missing: {', '.join(still_missing)}")
            print("   Please run: ./install_dependencies.sh")
            print("   Or restart kernel after running: ./install_dependencies.sh")
        else:
            print("\n✓ All core packages now available")
    else:
        print("✓ Core packages loaded")

    # Import core packages (now that they're installed)
    # Declare as global to make them available outside the function
    global np, requests, PDB, PDBIO
    try:
        import numpy as np
        import requests
        from Bio import PDB
        from Bio.PDB import PDBIO
        print("✓ Core modules imported")
    except ImportError as e:
        print(f"⚠ Import error: {e}")
        print("   Please restart kernel and run: ./install_dependencies.sh")
        return False

    # Check for external tools
    print("\n" + divider)
    print("External Tools Check")
    print(divider)

    # Check Rosetta
    rosetta_found = False
    rosetta_bin_path = Path("rosetta/source/bin")
    if rosetta_bin_path.exists():
        # Check for Rosetta binaries (they have .linuxgccrelease extension)
        rosetta_binaries = list(rosetta_bin_path.glob("*.linuxgccrelease"))
        if rosetta_binaries:
            # Check for common Rosetta applications
            common_apps = ["relax", "rosetta_scripts", "docking_protocol", "fixbb"]
            found_apps = []
            for app in common_apps:
                if any(app in str(bin_path) for bin_path in rosetta_binaries):
                    found_apps.append(app)
            
            if found_apps:
                print(f"✓ Rosetta (local: rosetta/source/bin)")
                print(f"   Found applications: {', '.join(found_apps)}")
                rosetta_found = True
            else:
                print("✓ Rosetta binaries found (local: rosetta/source/bin)")
                rosetta_found = True
        else:
            print("⚠ Rosetta source found but binaries not built")
            print("   Build with: ./install_rosetta.sh")
    elif Path("rosetta").exists():
        print("⚠ Rosetta directory found but binaries not built")
        print("   Build with: ./install_rosetta.sh")
    else:
        # Check if Rosetta is in PATH
        if subprocess.run(["which", "rosetta_scripts"], capture_output=True).returncode == 0 or \
           subprocess.run(["which", "relax"], capture_output=True).returncode == 0:
            print("✓ Rosetta (in PATH)")
            rosetta_found = True
        else:
            print("⚠ Rosetta not found (optional, used for structure refinement)")
            print("   Install with: ./install_rosetta.sh")

    print(divider)
    return True

# Call the setup function
setup_environment_and_check_dependencies()

In [ ]:
# AlphaFold 4 / ColabFold Peptide Structure Prediction
# Predict peptide 3D structure from amino acid sequence using AlphaFold
# Note: All imports are in Cell 0 above

def predict_peptide_structure_alphafold(sequence, output_dir=".", method="colabfold_api"):
    """
    Predict peptide structure using AlphaFold 4 / ColabFold.
    
    Parameters:
    - sequence: Amino acid sequence (single letter code, e.g., "YPFPGP")
    - output_dir: Directory to save output files
    - method: "colabfold_api" (default) or "colabfold_local" or "alphafold_db"
    
    Returns:
    - Path to predicted PDB file, or None if prediction fails
    """
    sequence = sequence.upper().strip()
    
    # Validate sequence
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if not all(aa in valid_aa for aa in sequence):
        invalid = [aa for aa in sequence if aa not in valid_aa]
        print(f"⚠ Invalid amino acids in sequence: {set(invalid)}")
        return None
    
    if len(sequence) < 5:
        print("⚠ Sequence too short (minimum 5 amino acids)")
        return None
    
    if len(sequence) > 2000:
        print("⚠ Sequence too long (maximum 2000 amino acids for peptides)")
        return None
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    output_pdb = output_path / f"{sequence}.pdb"
    
    if method == "colabfold_api":
        return _predict_colabfold_api(sequence, output_pdb)
    elif method == "colabfold_local":
        return _predict_colabfold_local(sequence, output_pdb)
    elif method == "alphafold_db":
        return _predict_alphafold_db(sequence, output_pdb)
    else:
        print(f"⚠ Unknown method: {method}")
        return None


def _predict_colabfold_api(sequence, output_pdb):
    """Predict using ESMFold API (free, fast, requires internet)"""
    try:
        import requests
        
        print(f"🔬 Predicting structure for sequence: {sequence}")
        print(f"   Length: {len(sequence)} amino acids")
        print(f"   Using ESMFold API (Meta AI - fast alternative to AlphaFold)...")
        
        # ESMFold API endpoint (free, fast alternative to AlphaFold)
        # Note: For AlphaFold 4 specifically, use ColabFold locally or AlphaFold Server
        api_url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
        
        print("   Submitting sequence to ESMFold API...")
        response = requests.post(api_url, data=sequence, timeout=120)
        
        if response.status_code == 200:
            # Save PDB file
            with open(output_pdb, 'w') as f:
                f.write(response.text)
            print(f"✓ Structure predicted and saved to: {output_pdb}")
            return str(output_pdb)
        else:
            print(f"⚠ API request failed with status {response.status_code}")
            print(f"   Response: {response.text[:200]}")
            print("   Trying alternative method...")
            return _predict_colabfold_local(sequence, output_pdb)
            
    except ImportError:
        print("⚠ requests library not available. Install with: pip install requests")
        return None
    except Exception as e:
        print(f"⚠ Error calling ESMFold API: {e}")
        print("   Trying alternative method...")
        return _predict_colabfold_local(sequence, output_pdb)


def _predict_colabfold_local(sequence, output_pdb):
    """Predict using local ColabFold installation"""
    try:
        import subprocess
        
        print(f"🔬 Predicting structure using local ColabFold...")
        
        # Check if colabfold_batch is available
        colabfold_cmd = "colabfold_batch"
        result = subprocess.run(["which", colabfold_cmd], capture_output=True)
        
        if result.returncode != 0:
            print("⚠ ColabFold not found locally")
            print("   Install with: pip install colabfold")
            print("   Or use method='colabfold_api' for API-based prediction")
            return None
        
        # Create temporary FASTA file
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as tmp_fasta:
            tmp_fasta.write(f">peptide\n{sequence}\n")
            tmp_fasta_path = tmp_fasta.name
        
        try:
            # Run ColabFold
            cmd = [colabfold_cmd, tmp_fasta_path, str(output_pdb.parent)]
            print(f"   Running: {' '.join(cmd)}")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
            
            if result.returncode == 0:
                # Find the output PDB file
                predicted_files = list(output_pdb.parent.glob(f"*{sequence}*.pdb"))
                if predicted_files:
                    predicted_file = predicted_files[0]
                    if predicted_file != output_pdb:
                        import shutil
                        shutil.copy(predicted_file, output_pdb)
                    print(f"✓ Structure predicted and saved to: {output_pdb}")
                    return str(output_pdb)
                else:
                    print("⚠ Output PDB file not found")
                    return None
            else:
                print(f"⚠ ColabFold failed: {result.stderr}")
                return None
        finally:
            # Clean up temp file
            if os.path.exists(tmp_fasta_path):
                os.unlink(tmp_fasta_path)
                
    except Exception as e:
        print(f"⚠ Error running local ColabFold: {e}")
        return None


def _predict_alphafold_db(sequence, output_pdb):
    """Try to fetch from AlphaFold Database if available"""
    try:
        import requests
        
        print(f"🔬 Searching AlphaFold Database for sequence...")
        
        # AlphaFold Database API
        # Note: This searches for exact matches in the database
        # For custom peptides, use ColabFold instead
        
        # Generate a unique identifier (hash of sequence)
        import hashlib
        seq_hash = hashlib.md5(sequence.encode()).hexdigest()
        
        # Try to fetch from AlphaFold DB (this is a simplified example)
        # In practice, you'd need to use the actual AlphaFold DB API
        print("⚠ AlphaFold Database lookup not fully implemented")
        print("   Use method='colabfold_api' for custom peptide prediction")
        return None
        
    except Exception as e:
        print(f"⚠ Error accessing AlphaFold Database: {e}")
        return None


# predicted_pdb = predict_peptide_structure_alphafold("YPFPGP", method="colabfold_api")
# if predicted_pdb:
#     print(f"Predicted structure saved to: {predicted_pdb}")


# Predict Peptide Structure Using AlphaFold 4

Use AlphaFold 4 / ColabFold to predict 3D structure from peptide sequence


In [ ]:
# Alternative: Predict multiple peptides or batch processing

def predict_multiple_peptides(sequences, output_dir="alphafold_predictions"):
    """
    Predict structures for multiple peptide sequences.
    
    Parameters:
    - sequences: List of amino acid sequences
    - output_dir: Directory to save all predictions
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    results = {}
    
    for i, seq in enumerate(sequences, 1):
        print(f"\n[{i}/{len(sequences)}] Processing: {seq}")
        predicted = predict_peptide_structure_alphafold(
            seq,
            output_dir=output_dir,
            method="colabfold_api"
        )
        results[seq] = predicted
        
        # Add a small delay between requests to avoid rate limiting
        if i < len(sequences):
            time.sleep(2)
    
    print("\n" + "=" * 60)
    print("Batch Prediction Summary")
    print("=" * 60)
    for seq, pdb_file in results.items():
        status = "✓" if pdb_file else "✗"
        print(f"{status} {seq}: {pdb_file or 'Failed'}")
    
    return results

# Helper function to extract peptides from FASTA file
def extract_peptides_from_fasta(fasta_file, num_peptides=5):
    """
    Extract peptide sequences from a FASTA file.
    
    Parameters:
    - fasta_file: Path to FASTA file
    - num_peptides: Number of peptides to extract (default: 5)
    
    Returns:
    - List of peptide sequences
    """
    peptides = []
    try:
        with open(fasta_file, 'r') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('>'):
                    peptides.append(line)
                    if len(peptides) >= num_peptides:
                        break
        print(f"✓ Extracted {len(peptides)} peptides from {fasta_file}")
        return peptides
    except FileNotFoundError:
        print(f"⚠ File not found: {fasta_file}")
        return []
    except Exception as e:
        print(f"⚠ Error reading file: {e}")
        return []

# Example: Extract first 5 peptides from intestinal unique peptides.txt
peptide_file = "intestinal unique peptides.txt"
peptide_sequences = extract_peptides_from_fasta(peptide_file, num_peptides=5)

# Display the extracted peptides
print(f"\nExtracted {len(peptide_sequences)} peptides:")
for i, seq in enumerate(peptide_sequences, 1):
    print(f"  {i}. {seq} (length: {len(seq)} amino acids)")

# Uncomment to run batch prediction:
#batch_results = predict_multiple_peptides(peptide_sequences)


# Rosetta Suite Docking
Using Rosetta for advanced peptide-protein docking and structure refinement

In [ ]:
# SSH/HPC Remote Execution Support
# Execute Rosetta docking on remote HPC cluster via SSH

import subprocess
import tempfile
import shutil

# HPC Configuration
HPC_CONFIG = {
    "host": "hpc.cqls.oregonstate.edu",
    "user": None,  # Will use current user or detect from SSH config
    "key": None,  # Will use default SSH key
    "remote_workdir": "~/peptide-md-docking",  # Remote working directory
    "ssh_verbose": "-vvv",  # Verbose SSH output
}

def setup_hpc_connection(host=None, user=None, key=None, remote_workdir=None):
    """
    Configure HPC connection parameters.
    
    Parameters:
    - host: HPC hostname (default: hpc.cqls.oregonstate.edu)
    - user: SSH username (default: auto-detect)
    - key: Path to SSH private key (default: use default SSH key)
    - remote_workdir: Remote working directory (default: ~/peptide-md-docking)
    """
    if host:
        HPC_CONFIG["host"] = host
    if user:
        HPC_CONFIG["user"] = user
    if key:
        HPC_CONFIG["key"] = key
    if remote_workdir:
        HPC_CONFIG["remote_workdir"] = remote_workdir
    
    print("=" * 60)
    print("HPC Configuration")
    print("=" * 60)
    print(f"Host: {HPC_CONFIG['host']}")
    print(f"User: {HPC_CONFIG['user'] or '(auto-detect)'}")
    print(f"Remote workdir: {HPC_CONFIG['remote_workdir']}")
    print("=" * 60)

def _build_ssh_command(base_cmd, remote_cmd=None):
    """Build SSH command with proper options"""
    ssh_cmd = ["ssh", HPC_CONFIG["ssh_verbose"]]
    
    # Add identity file if specified
    if HPC_CONFIG["key"]:
        ssh_cmd.extend(["-i", HPC_CONFIG["key"]])
    
    # Add user@host
    if HPC_CONFIG["user"]:
        ssh_target = f"{HPC_CONFIG['user']}@{HPC_CONFIG['host']}"
    else:
        ssh_target = HPC_CONFIG["host"]
    
    ssh_cmd.append(ssh_target)
    
    # Add remote command if provided
    if remote_cmd:
        if isinstance(remote_cmd, str):
            ssh_cmd.append(remote_cmd)
        else:
            ssh_cmd.extend(remote_cmd)
    
    return ssh_cmd

def _build_scp_command(local_path, remote_path, direction="to"):
    """Build SCP command for file transfer
    
    direction: "to" (local -> remote) or "from" (remote -> local)
    """
    scp_cmd = ["scp", "-r"]  # -r for recursive
    
    # Add verbose flag
    if HPC_CONFIG["ssh_verbose"]:
        scp_cmd.append(HPC_CONFIG["ssh_verbose"])
    
    # Add identity file if specified
    if HPC_CONFIG["key"]:
        scp_cmd.extend(["-i", HPC_CONFIG["key"]])
    
    # Build source and destination
    if HPC_CONFIG["user"]:
        remote_str = f"{HPC_CONFIG['user']}@{HPC_CONFIG['host']}:"
    else:
        remote_str = f"{HPC_CONFIG['host']}:"
    
    if direction == "to":
        scp_cmd.append(str(local_path))
        scp_cmd.append(remote_str + str(remote_path))
    else:  # from
        scp_cmd.append(remote_str + str(remote_path))
        scp_cmd.append(str(local_path))
    
    return scp_cmd

def _test_hpc_connection():
    """Test SSH connection to HPC"""
    print("Testing HPC connection...")
    test_cmd = _build_ssh_command("ssh", "echo 'Connection successful'")
    result = subprocess.run(test_cmd, capture_output=True, text=True, timeout=30)
    
    if result.returncode == 0:
        print("✓ HPC connection successful")
        return True
    else:
        print(f"⚠ HPC connection failed: {result.stderr[:200]}")
        return False

def _ensure_remote_directory(remote_dir):
    """Ensure remote directory exists"""
    mkdir_cmd = _build_ssh_command("ssh", f"mkdir -p {remote_dir}")
    subprocess.run(mkdir_cmd, capture_output=True, timeout=30)

def _upload_file_to_hpc(local_file, remote_file):
    """Upload file to HPC cluster"""
    try:
        # Ensure remote directory exists
        remote_dir = str(Path(remote_file).parent)
        _ensure_remote_directory(remote_dir)
        
        # Upload file
        scp_cmd = _build_scp_command(local_file, remote_file, "to")
        result = subprocess.run(scp_cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            return True
        else:
            print(f"⚠ Upload failed: {result.stderr[:300]}")
            return False
    except Exception as e:
        print(f"⚠ Error uploading file: {e}")
        return False

def _download_file_from_hpc(remote_file, local_file):
    """Download file from HPC cluster"""
    try:
        # Ensure local directory exists
        local_dir = Path(local_file).parent
        local_dir.mkdir(parents=True, exist_ok=True)
        
        # Download file
        scp_cmd = _build_scp_command(local_file, remote_file, "from")
        result = subprocess.run(scp_cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            return True
        else:
            print(f"⚠ Download failed: {result.stderr[:300]}")
            return False
    except Exception as e:
        print(f"⚠ Error downloading file: {e}")
        return False

def _execute_remote_command(remote_cmd, timeout=3600):
    """Execute command on remote HPC cluster"""
    ssh_cmd = _build_ssh_command("ssh", remote_cmd)
    
    try:
        print(f"Executing on HPC: {' '.join(remote_cmd[:3])}...")
        result = subprocess.run(ssh_cmd, capture_output=True, text=True, timeout=timeout)
        return result
    except subprocess.TimeoutExpired:
        print("⚠ Command timed out")
        return None
    except Exception as e:
        print(f"⚠ Error executing remote command: {e}")
        return None

# Test connection on import (optional, comment out if not needed)
# _test_hpc_connection()

print("=" * 60)
print("HPC/SSH Remote Execution Support")
print("=" * 60)
print("✓ SSH utilities loaded")
print(f"Default HPC: {HPC_CONFIG['host']}")
print("\nTo configure HPC connection:")
print("  setup_hpc_connection(host='hpc.cqls.oregonstate.edu', user='your_username')")
print("=" * 60)



In [ ]:
# Rosetta Suite Integration
# Comprehensive Rosetta tool detection and integration


def _find_rosetta_binaries():
    """
    Find Rosetta binaries from multiple possible locations:
    1. System PATH
    2. Local rosetta/source/bin directory (from install_rosetta.sh)
    3. Conda environment
    """
    rosetta_bin = {}
    rosetta_paths = []
    
    # Check system PATH
    for cmd in ["rosetta_scripts", "relax", "docking_protocol", "fixbb"]:
        result = subprocess.run(["which", cmd], capture_output=True)
        if result.returncode == 0:
            rosetta_bin[cmd] = result.stdout.decode().strip()
            rosetta_paths.append(Path(rosetta_bin[cmd]).parent)
    
    # Check local rosetta installation (from install_rosetta.sh)
    # Rosetta structure: rosetta/source/bin (binaries built here after ./install_rosetta.sh)
    rosetta_dir = Path("rosetta")
    local_rosetta_bin = Path("rosetta/source/bin")
    
    # Verify rosetta directory exists
    if rosetta_dir.exists() and rosetta_dir.is_dir():
        # Check if binaries directory exists
        if local_rosetta_bin.exists() and local_rosetta_bin.is_dir():
            # Look for actual Rosetta binaries (not just any executable)
            rosetta_binary_names = ["relax", "rosetta_scripts", "docking_protocol", "fixbb", 
                                    "rosetta_scripts.linuxgccrelease", "relax.linuxgccrelease"]
            for cmd_file in local_rosetta_bin.glob("*"):
                if cmd_file.is_file() and os.access(cmd_file, os.X_OK):
                    cmd_name = cmd_file.name
                    # Check if it's a Rosetta binary (by name or if directory has Rosetta binaries)
                    if cmd_name in rosetta_binary_names or any(rosetta_name in cmd_name for rosetta_name in ["rosetta", "relax", "docking", "fixbb"]):
                        if cmd_name not in rosetta_bin:
                            rosetta_bin[cmd_name] = str(cmd_file.absolute())
                            rosetta_paths.append(local_rosetta_bin)
                            
                            # Also add simplified key (without extension) for easier lookup
                            # e.g., "relax.linuxgccrelease" -> also add "relax"
                            if ".linuxgccrelease" in cmd_name:
                                base_name = cmd_name.replace(".linuxgccrelease", "")
                                # Remove .default if present
                                if base_name.endswith(".default"):
                                    base_name = base_name.replace(".default", "")
                                # Only add if not already present (prefer non-extension version)
                                if base_name not in rosetta_bin:
                                    rosetta_bin[base_name] = str(cmd_file.absolute())
    
    # Check conda environment
    if "CONDA_PREFIX" in os.environ:
        conda_bin = Path(os.environ["CONDA_PREFIX"]) / "bin"
        if conda_bin.exists():
            for cmd in ["rosetta_scripts", "relax"]:
                conda_cmd = conda_bin / cmd
                if conda_cmd.exists() and cmd not in rosetta_bin:
                    rosetta_bin[cmd] = str(conda_cmd)
                    rosetta_paths.append(conda_bin)
    
    return rosetta_bin, rosetta_paths

def get_rosetta_command(cmd_name):
    """Get full path to a Rosetta command"""
    # First try exact match
    if cmd_name in rosetta_commands:
        return rosetta_commands[cmd_name]
    
    # Try with .linuxgccrelease extension
    if f"{cmd_name}.linuxgccrelease" in rosetta_commands:
        return rosetta_commands[f"{cmd_name}.linuxgccrelease"]
    
    # Try with .default.linuxgccrelease extension
    if f"{cmd_name}.default.linuxgccrelease" in rosetta_commands:
        return rosetta_commands[f"{cmd_name}.default.linuxgccrelease"]
    
    # Try to find any key that starts with cmd_name
    for key in rosetta_commands.keys():
        if key.startswith(cmd_name):
            return rosetta_commands[key]
    
    return None

# Detect Rosetta installation
rosetta_commands, rosetta_paths = _find_rosetta_binaries()
ROSETTA_AVAILABLE = len(rosetta_commands) > 0

# Print status
print("=" * 60)
print("Rosetta Suite Detection")
print("=" * 60)

if ROSETTA_AVAILABLE:
    print(f"✓ Rosetta found ({len(rosetta_commands)} command(s))")
    if rosetta_paths:
        unique_paths = list(set(str(p) for p in rosetta_paths))
        print(f"  Location(s): {', '.join(unique_paths[:2])}")
    
    # Show only relevant tools for docking workflow
    relevant_tools = ["relax", "rosetta_scripts", "docking_protocol", "fixbb"]
    found_relevant = []
    
    print("\nRelevant Rosetta tools for docking workflow:")
    for tool in relevant_tools:
        if tool in rosetta_commands:
            print(f"  ✓ {tool}")
            found_relevant.append(tool)
        else:
            # Check for variants with extensions
            variants = [k for k in rosetta_commands.keys() if k.startswith(tool) and tool in k]
            if variants:
                # Prefer the one without extension, or first variant
                preferred = [v for v in variants if not v.endswith('.linuxgccrelease')]
                if preferred:
                    print(f"  ✓ {preferred[0]}")
                    found_relevant.append(tool)
                else:
                    print(f"  ✓ {variants[0]}")
                    found_relevant.append(tool)
    
    if len(found_relevant) < len(relevant_tools):
        missing = [t for t in relevant_tools if t not in found_relevant]
        if missing:
            print(f"\n  ⚠ Note: Some tools not found: {', '.join(missing)}")
    
    # Set ROSETTA_BIN_PATH for subprocess calls
    if rosetta_paths:
        ROSETTA_BIN_PATH = str(rosetta_paths[0])
        os.environ["ROSETTA_BIN_PATH"] = ROSETTA_BIN_PATH
    else:
        ROSETTA_BIN_PATH = None
else:
    print("⚠ Rosetta not found")
    print("\nInstallation options:")
    print("  1. From GitHub: ./install_rosetta.sh")
    print("  2. Via conda: conda install -c conda-forge rosetta")
    print("  3. Manual: See INSTALL_MANUAL.md")
    print("\nNote: Rosetta is required for this workflow")
    ROSETTA_BIN_PATH = None

print("=" * 60)


In [ ]:
# Rosetta Peptide-Protein Docking
# Use Rosetta for advanced peptide docking against receptor

def rosetta_peptide_docking(receptor_pdb, peptide_pdb, output_dir="rosetta_docking",
                            nstruct=10, relax=True):
    """
    Perform peptide-protein docking using Rosetta.
    
    Parameters:
    - receptor_pdb: Path to receptor PDB file
    - peptide_pdb: Path to peptide PDB file
    - output_dir: Directory to save results
    - nstruct: Number of structures to generate (default: 10)
    - relax: Whether to relax structures after docking (default: True)
    
    Returns:
    - Path to best docked structure, or None if docking fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Get Rosetta command using helper function
    rosetta_scripts = get_rosetta_command("rosetta_scripts")
    if not rosetta_scripts:
        print("⚠ rosetta_scripts not found")
        print("   Available commands:", list(rosetta_commands.keys()))
        return None
    
    print("=" * 60)
    print("Rosetta Peptide-Protein Docking")
    print("=" * 60)
    print(f"Receptor: {receptor_pdb}")
    print(f"Peptide:  {peptide_pdb}")
    print(f"Output:   {output_dir}")
    print(f"Structures: {nstruct}")
    
    # Create Rosetta XML script for peptide docking
    # Generate Rosetta XML script inline
    # Using a valid XML format for peptide-protein docking
    xml_content = """<?xml version="1.0"?>
    <ROSETTASCRIPTS>
        <SCOREFXNS>
            <ScoreFunction name="ref2015" weights="ref2015"/>
            <ScoreFunction name="ref2015_cart" weights="ref2015_cart"/>
        </SCOREFXNS>
        
        <MOVERS>
            <Docking name="docking" fullatom="1" local_refine="1" score_high="ref2015"/>
            <FastRelax name="relax" scorefxn="ref2015_cart" />
        </MOVERS>
        
        <PROTOCOLS>
            <Add mover_name="docking"/>
            <Add mover_name="relax"/>
        </PROTOCOLS>
    </ROSETTASCRIPTS>
    """
    
    # Write XML to file for Rosetta
    xml_script = output_path / "peptide_docking.xml"
    with open(xml_script, "w") as f:
        f.write(xml_content)
    
    # Prepare input files
    # Combine receptor and peptide into a single PDB for docking
    combined_pdb = output_path / "input_complex.pdb"
    receptor_chains, peptide_chain = _combine_pdb_files(receptor_pdb, peptide_pdb, combined_pdb)
    
    if not receptor_chains or not peptide_chain:
        print("⚠ Error: Could not determine chain IDs for docking partners")
        return None
    
    # Format partners string: receptor_chains_peptide_chain (e.g., "A_B" or "ABC_D")
    partners_str = f"{''.join(receptor_chains)}_{peptide_chain}"
    print(f"Docking partners: {partners_str}")
    
    # Run Rosetta docking
    try:
        cmd = [
            rosetta_scripts,
            "-parser:protocol", str(xml_script),
            "-s", str(combined_pdb),
            "-partners", partners_str,  # Specify docking partners
            "-nstruct", str(nstruct),
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "docking_scores.sc"),
            "-ex1", "-ex2",  # Extra rotamer sampling
            "-use_input_sc",  # Use input side chain conformations
            "-flip_HNQ",  # Flip Asn, Gln, His
            "-no_optH", "false"  # Optimize hydrogens
        ]
        
        if relax:
            cmd.extend(["-relax:fast"])  # Fast relaxation
        
        print(f"\nRunning: {' '.join(cmd[:5])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            # Find best structure based on score
            best_structure = _find_best_rosetta_structure(output_path)
            if best_structure:
                print(f"\n✓ Docking completed successfully!")
                print(f"Best structure: {best_structure}")
                return str(best_structure)
            else:
                print("\n⚠ Docking completed but best structure not found")
                return None
        else:
            print(f"\n⚠ Rosetta docking failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running Rosetta: {e}")
        return None


def _combine_pdb_files(receptor_pdb, peptide_pdb, output_pdb):
    """
    Combine receptor and peptide PDB files.
    
    Returns:
    - tuple: (receptor_chain_ids, peptide_chain_id) or (None, None) on error
    """
    try:
        from Bio import PDB
        
        parser = PDB.PDBParser(QUIET=True)
        receptor_structure = parser.get_structure("receptor", receptor_pdb)
        peptide_structure = parser.get_structure("peptide", peptide_pdb)
        
        # Add peptide as a new chain to receptor
        receptor_model = list(receptor_structure.get_models())[0]
        peptide_model = list(peptide_structure.get_models())[0]
        
        # Get receptor chain IDs
        receptor_chains = [chain.id for chain in receptor_model]
        
        # Get next chain ID for peptide
        new_chain_id = chr(ord('A') + len(receptor_chains))
        
        # Create new chain for peptide
        new_chain = PDB.Chain.Chain(new_chain_id)
        for chain in peptide_model:
            for residue in chain:
                new_chain.add(residue.copy())
        
        receptor_model.add(new_chain)
        
        # Save combined structure
        io = PDB.PDBIO()
        io.set_structure(receptor_structure)
        io.save(str(output_pdb))
        
        return (receptor_chains, new_chain_id)
    except Exception as e:
        print(f"⚠ Error combining PDB files: {e}")
        return (None, None)


def _find_best_rosetta_structure(output_path):
    """Find best structure from Rosetta docking results"""
    score_file = output_path / "docking_scores.sc"
    
    if not score_file.exists():
        # Look for any PDB files
        pdb_files = list(output_path.glob("*.pdb"))
        if pdb_files:
            return pdb_files[0]
        return None
    
    try:
        best_score = float('inf')
        best_structure = None
        
        with open(score_file, 'r') as f:
            for line in f:
                if line.startswith("SCORE:"):
                    parts = line.split()
                    if "total_score" in parts:
                        score_idx = parts.index("total_score")
                        if score_idx + 1 < len(parts):
                            try:
                                score = float(parts[score_idx + 1])
                                if score < best_score:
                                    best_score = score
                                    # Find corresponding PDB file
                                    if "description" in parts:
                                        desc_idx = parts.index("description")
                                        if desc_idx + 1 < len(parts):
                                            desc = parts[desc_idx + 1]
                                            pdb_file = output_path / f"{desc}.pdb"
                                            if pdb_file.exists():
                                                best_structure = pdb_file
                            except (ValueError, IndexError):
                                continue
        
        return best_structure
        
    except Exception as e:
        print(f"⚠ Error parsing score file: {e}")
        # Fallback: return first PDB file
        pdb_files = list(output_path.glob("*.pdb"))
        return pdb_files[0] if pdb_files else None


def rosetta_relax(pdb_file, output_pdb=None, nstruct=5):
    """
    Relax/refine structure using Rosetta relax application.
    This is simpler than full docking and useful for structure refinement.
    
    Parameters:
    - pdb_file: Path to input PDB file
    - output_pdb: Path to output PDB file (default: auto-generated)
    - nstruct: Number of relaxed structures to generate (default: 5)
    
    Returns:
    - Path to best relaxed structure, or None if relaxation fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    relax_cmd = get_rosetta_command("relax")
    if not relax_cmd:
        print("⚠ Rosetta 'relax' command not found")
        print("   Available commands:", list(rosetta_commands.keys()))
        return None
    
    if output_pdb is None:
        output_pdb = Path(pdb_file).stem + "_relaxed.pdb"
    
    output_path = Path(output_pdb).parent
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta Structure Relaxation")
    print("=" * 60)
    print(f"Input:  {pdb_file}")
    print(f"Output: {output_pdb}")
    print(f"Structures: {nstruct}")
    
    try:
        cmd = [
            relax_cmd,
            "-s", str(pdb_file),
            "-nstruct", str(nstruct),
            "-relax:fast",
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "relax_scores.sc")
        ]
        
        print(f"\nRunning: {' '.join(cmd[:3])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            # Find best structure
            score_file = output_path / "relax_scores.sc"
            if score_file.exists():
                best_structure = _find_best_rosetta_structure(output_path)
                if best_structure:
                    print(f"\n✓ Relaxation completed!")
                    print(f"Best structure: {best_structure}")
                    return str(best_structure)
            
            print("\n✓ Relaxation completed (check output directory)")
            return str(output_path)
        else:
            print(f"\n⚠ Rosetta relax failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running Rosetta relax: {e}")
        return None


def rosetta_docking_protocol(receptor_pdb, ligand_pdb, output_dir="rosetta_docking_protocol",
                             nstruct=10, docking_method="local_refine"):
    """
    Use Rosetta docking_protocol for protein-protein/peptide-protein docking.
    This is more robust than rosetta_scripts for docking.
    
    Parameters:
    - receptor_pdb: Path to receptor PDB file
    - ligand_pdb: Path to ligand/peptide PDB file
    - output_dir: Directory to save results
    - nstruct: Number of structures to generate
    - docking_method: "local_refine" (default) or "perturb"
    
    Returns:
    - Path to best docked structure, or None if docking fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    docking_protocol = get_rosetta_command("docking_protocol")
    if not docking_protocol:
        print("⚠ docking_protocol not found")
        print("   Falling back to rosetta_scripts...")
        return rosetta_peptide_docking(receptor_pdb, ligand_pdb, output_dir, nstruct)
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta Docking Protocol")
    print("=" * 60)
    print(f"Receptor: {receptor_pdb}")
    print(f"Ligand:   {ligand_pdb}")
    print(f"Method:   {docking_method}")
    print(f"Structures: {nstruct}")
    
    try:
        cmd = [
            docking_protocol,
            "-s", str(receptor_pdb), str(ligand_pdb),
            "-nstruct", str(nstruct),
            "-docking", docking_method,
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "docking_scores.sc")
        ]
        
        print(f"\nRunning: {' '.join(cmd[:5])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            best_structure = _find_best_rosetta_structure(output_path)
            if best_structure:
                print(f"\n✓ Docking completed!")
                print(f"Best structure: {best_structure}")
                return str(best_structure)
            else:
                print("\n✓ Docking completed (check output directory)")
                return str(output_path)
        else:
            print(f"\n⚠ Docking failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running docking_protocol: {e}")
        return None


def rosetta_fixbb(pdb_file, output_pdb=None, resfile=None):
    """
    Use Rosetta fixbb (fix backbone) to redesign side chains.
    Useful for structure optimization.
    
    Parameters:
    - pdb_file: Path to input PDB file
    - output_pdb: Path to output PDB file
    - resfile: Optional resfile for specific residue design
    
    Returns:
    - Path to output structure, or None if fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available")
        return None
    
    fixbb_cmd = get_rosetta_command("fixbb")
    if not fixbb_cmd:
        print("⚠ fixbb not found")
        return None
    
    if output_pdb is None:
        output_pdb = Path(pdb_file).stem + "_fixbb.pdb"
    
    output_path = Path(output_pdb).parent
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta FixBB (Side Chain Design)")
    print("=" * 60)
    print(f"Input:  {pdb_file}")
    print(f"Output: {output_pdb}")
    
    try:
        cmd = [fixbb_cmd, "-s", str(pdb_file), "-out:path:pdb", str(output_path)]
        if resfile:
            cmd.extend(["-resfile", str(resfile)])
        
        print(f"\nRunning: {' '.join(cmd[:3])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"\n✓ FixBB completed!")
            return str(output_pdb)
        else:
            print(f"\n⚠ FixBB failed: {result.stderr[:300]}")
            return None
            
    except Exception as e:
        print(f"\n⚠ Error: {e}")
        return None


# Print Rosetta status and usage
print("\n" + "=" * 60)
print("Rosetta Functions Available")
print("=" * 60)

if ROSETTA_AVAILABLE:
    print("\n✓ Rosetta is ready!")
    print("\nAvailable functions:")
    print("1. rosetta_relax() - Structure relaxation/refinement")
    print("2. rosetta_peptide_docking() - Peptide-protein docking (rosetta_scripts)")
    print("3. rosetta_docking_protocol() - Docking using docking_protocol")
    print("4. rosetta_fixbb() - Side chain redesign")
    print("\nExample usage:")
    print("  # Relax structure")
    print("  relaxed = rosetta_relax('3fxi.pdb', nstruct=5)")
    print("  ")
    print("  # Dock peptide to receptor")
    print("  docked = rosetta_peptide_docking('3fxi.pdb', 'peptide.pdb', nstruct=10)")
else:
    print("\n⚠ Rosetta not installed")
    print("\nTo install Rosetta:")
    print("  Option 1: ./install_rosetta.sh (from GitHub)")
    print("  Option 2: conda install -c conda-forge rosetta")
    print("  Option 3: See INSTALL_MANUAL.md")
    print("\nNote: Rosetta is required for this workflow")

print("=" * 60)


In [ ]:
#relaxed = rosetta_relax('3fxi.pdb', nstruct=1)
#docked = rosetta_peptide_docking(relaxed, 'alphafold_predictions/ACDEFGHIKLMNPQRSTVWY.pdb', nstruct=1)

In [64]:

# Dock peptide to receptor
docked = rosetta_peptide_docking('3fxi_0001.pdb', 'alphafold_predictions/ACDEFGHIKLMNPQRSTVWY_alphafold.pdb', nstruct=1)


KeyboardInterrupt: 